# Student Performance — Exploratory Data Analysis

This notebook explores the Student Performance dataset (student-mat.csv).  
Goal: Understand the data before building a predictive model for the final grade (G3).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/student/student-mat.csv", sep=";")
print("Dataset loaded successfully.")

## 2. Dataset Overview

### 2.1 Dimensions

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

### 2.2 First 5 Rows

In [ ]:
df.head()

### 2.3 Last 5 Rows

In [ ]:
df.tail()

### 2.4 Column Names

In [ ]:
print(df.columns.tolist())

### 2.5 Data Types

In [ ]:
df.info()

### 2.6 Descriptive Statistics

In [ ]:
df.describe()

## 3. Data Quality

### 3.1 Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")

**Why this matters:** Missing values can bias models. Since there are none, we can proceed without imputation.

### 3.2 Duplicate Rows

In [ ]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")

**Why duplicates matter:** They give undue weight to repeated observations and can cause data leakage between train and test sets.

## 4. Target Variable Analysis (G3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["G3"], bins=20, edgecolor="black", color="steelblue")
axes[0].set_xlabel("Final Grade (G3)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram of G3")

axes[1].boxplot(df["G3"])
axes[1].set_ylabel("G3")
axes[1].set_title("Box Plot of G3")
axes[1].grid(True)

df["G3"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color="coral")
axes[2].set_xlabel("G3")
axes[2].set_ylabel("Count")
axes[2].set_title("Grade Distribution")

plt.tight_layout()
plt.savefig("../models/g3_distribution.png", dpi=150)
plt.show()

In [ ]:
print(f"Mean G3: {df['G3'].mean():.2f}")
print(f"Median G3: {df['G3'].median():.2f}")
print(f"Std G3: {df['G3'].std():.2f}")
print(f"Min G3: {df['G3'].min()}")
print(f"Max G3: {df['G3'].max()}")

**Observations:**  
- G3 ranges from 0 to 20 (Portuguese grading scale).  
- Distribution is roughly normal with a slight left skew.  
- Most students score between 8 and 14.  
- There are students with 0 (extreme low performance).

## 5. Numerical Feature Analysis

### 5.1 Histograms for All Numerical Features

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["G1", "G2", "G3"]]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=15, edgecolor="black", color="skyblue")
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Histograms of Numerical Features", fontsize=14)
plt.tight_layout()
plt.savefig("../models/numerical_histograms.png", dpi=150)
plt.show()

**What histograms reveal:**  
- **age:** Right-skewed — most students are 15–18.  
- **Medu/Fedu (parental education):** Peak at 4 (higher education).  
- **studytime:** Most study 2–4 hours/week (low).  
- **failures:** Strongly right-skewed — most have 0 failures.  
- **absences:** Right-skewed — most miss few days, some miss many.  
- **alcohol consumption (Dalc/Walc):** Most drink little on weekdays, more on weekends.

### 5.2 Box Plots

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col])
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Box Plots of Numerical Features", fontsize=14)
plt.tight_layout()
plt.savefig("../models/numerical_boxplots.png", dpi=150)
plt.show()

**Box plots reveal outliers:**  
- **absences** has extreme outliers (up to 75 absences).  
- **age** has students up to 22 (unusual for secondary school).  
- **Walc** has many high-consumption students (outliers at 4–5).

## 6. Categorical Feature Analysis

In [ ]:
cat_cols = df.select_dtypes(include=["object", "str"]).columns.tolist()
print("Categorical features:", cat_cols)

### 6.1 Bar Charts

In [ ]:
n_cat = len(cat_cols)
fig, axes = plt.subplots(5, 4, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(counts.index, counts.values, color="teal", edgecolor="black")
    axes[i].set_title(col, fontsize=11)
    axes[i].tick_params(axis="x", rotation=45, labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig("../models/categorical_barcharts.png", dpi=150)
plt.show()

**Key observations from categorical features:**  
- **school:** Most students attend GP (Gabriel Pereira).  
- **address:** Most live in urban areas (U).  
- **Mjob/Fjob:** Mother's jobs vary more than father's.  
- **higher:** Nearly all want higher education (important predictor).  
- **romantic:** About 1/3 are in a relationship.

## 7. Correlation Matrix

In [ ]:
# Encode categorical variables for correlation
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include=["object", "str"]).columns:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]

corr = df_encoded.corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of All Features", fontsize=14)
plt.tight_layout()
plt.savefig("../models/correlation_matrix.png", dpi=150)
plt.show()

In [ ]:
# Top correlations with G3
g3_corr = corr["G3"].drop("G3").sort_values(ascending=False)
print("Top 10 features correlated with G3:")
print(g3_corr.head(10))

**Correlation insights:**  
- **G1, G2** have the highest correlation with G3 — but we exclude them to avoid data leakage.  
- **failures** is strongly negatively correlated with G3 (−0.35) — more failures = lower grade.  
- **studytime** has a moderate positive correlation.  
- **absences** has a weak negative correlation.  
- **age** is slightly negatively correlated — older students tend to have lower grades.

## 8. Scatter Plots Against G3

In [ ]:
key_features = ["studytime", "failures", "absences", "age", "Medu", "goout", "Walc"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    axes[i].scatter(df[col], df["G3"], alpha=0.5, s=30, color="steelblue")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("G3")
    axes[i].set_title(f"{col} vs G3")

    # Add trend line
    m, b = np.polyfit(df[col], df["G3"], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, m * x_line + b, "r--", linewidth=1.5)

axes[7].set_visible(False)
plt.suptitle("Scatter Plots of Key Features vs Final Grade", fontsize=14)
plt.tight_layout()
plt.savefig("../models/scatter_plots.png", dpi=150)
plt.show()

**Scatter plot insights:**  
- **studytime vs G3:** Weak positive trend — more study hours, slightly higher grades.  
- **failures vs G3:** Clear negative trend — past failures strongly predict lower final grades.  
- **absences vs G3:** Weak negative trend — more absences, lower grades.  
- **age vs G3:** Slight negative trend — older students tend to have lower grades (possibly due to grade retention).  
- **Medu vs G3:** Mother's education shows a modest positive relationship with student grades.  
- **goout vs G3:** No strong relationship — socializing doesn't strongly affect grades.  
- **Walc vs G3:** Slight negative trend — higher weekend alcohol consumption correlates with lower grades.

## 9. Why Exclude G1 and G2?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df["G1"], df["G3"], alpha=0.5, s=30, color="green")
axes[0].set_xlabel("G1 (First Period Grade)")
axes[0].set_ylabel("G3 (Final Grade)")
axes[0].set_title("G1 vs G3")

axes[1].scatter(df["G2"], df["G3"], alpha=0.5, s=30, color="purple")
axes[1].set_xlabel("G2 (Second Period Grade)")
axes[1].set_ylabel("G3 (Final Grade)")
axes[1].set_title("G2 vs G3")

plt.tight_layout()
plt.savefig("../models/g1g2_vs_g3.png", dpi=150)
plt.show()

In [ ]:
print(f"Correlation G1-G3: {df['G1'].corr(df['G3']):.3f}")
print(f"Correlation G2-G3: {df['G2'].corr(df['G3']):.3f}")

**Why exclude G1 and G2?**  
- G1 and G2 are the student's grades from the first and second periods of the same school year.  
- They are *strongly* correlated with G3 (the final grade).  
- If we include them, the model learns to predict `G3 ≈ G2` (or `G3 ≈ G1`), which is trivial and useless.  
- **The goal** is to predict the final grade **before** the student takes any exams — using only demographic and behavioral features.  
- Including G1/G2 would cause **data leakage** — the model would have access to information that doesn't exist at prediction time.  
- This is why the prompt explicitly says to drop them.

## 10. EDA Summary

| Finding | Implication |
|---------|-------------|
| No missing values | No imputation needed |
| Some duplicates exist | Remove to avoid bias |
| G3 ranges 0–20, roughly normal | Regression approach is appropriate |
| failures has strong negative correlation with G3 | Key predictor |
| studytime has positive correlation | Expected — encourage more study |
| absences has negative correlation | Attendance matters |
| Parental education (Medu/Fedu) correlates with grades | Socioeconomic factors matter |
| G1/G2 are nearly perfect predictors | Must exclude to avoid leakage |
| Most students are 15–18 | Age range is narrow |
| Some outliers in absences | Consider winsorization |